# Testing: Synthetic Data

The `synthetic` module allows us to create artificial datasets with known velocities, extents etc. which we can then compare with those estimated by THUNER.

In [32]:
"""Synthetic data demo/test."""

%load_ext autoreload
%autoreload 2
import xarray as xr
from pathlib import Path
import shutil
import numpy as np
import thuner.data as data
import thuner.default as default
import thuner.track.track as track
import thuner.option as option
import thuner.analyze as analyze
import thuner.data.synthetic as synthetic
from thuner.utils import format_time
from thuner.log import setup_logger

logger = setup_logger(__name__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Geographic Coordinates

In [ ]:
# Set a flag for whether or not to remove existing output directories
remove_existing_outputs = True

# Parent directory for saving outputs
base_local = Path.home() / "THUNER_output"
start = "2005-11-13T00:00:00"
end = "2005-11-13T03:00:00"

output_parent = base_local / "runs/synthetic/geographic"

In [15]:
if output_parent.exists() and remove_existing_outputs:
    shutil.rmtree(output_parent)

In [37]:
options_directory = output_parent / "options"
options_directory.mkdir(parents=True, exist_ok=True)

# Create a grid
lat = np.arange(-14, -6 + 0.025, 0.025).tolist()
lon = np.arange(128, 136 + 0.025, 0.025).tolist()
grid_options = option.grid.GridOptions(name="geographic", latitude=lat, longitude=lon)
grid_options.to_json(options_directory / "grid.json")

# Initialize synthetic objects. Each is given a finite lifetime (30-120 min) and linear
# fade-in/out, so objects appear, intensify, weaken and disappear over the run.
starting_objects = []
for i in range(5):
    major = 3 * (7 + 4 * i)  # full axis length in km
    obj = synthetic.EllipsoidObject(
        time=start,
        center_latitude=np.mean(lat),
        center_longitude=lon[(i + 1) * len(lon) // 6],
        direction=-np.pi / 4 + i * np.pi / 8,
        speed=30 - 4 * i,
        major=major,
        minor=0.4 * major,
        orientation=0.25 * np.pi + i * np.pi / 8,
        life_time=120 + i * 30,
        fade_in_time=60,
        fade_out_time=60,
    )
    starting_objects.append(obj)
# Create data options dictionary. The objects are owned by a generator; FixedGenerator
# simply replays this fixed list (procedural generators are a future extension).
generator = synthetic.FixedGenerator(objects=starting_objects)
synthetic_options = data.synthetic.SyntheticOptions(generator=generator)
data_options = option.data.DataOptions(datasets=[synthetic_options])
data_options.to_json(options_directory / "data.json")

track_options = default.track.synthetic_track()
track_options.to_json(options_directory / "track.json")

# Create the display_options dictionary
visualize_options = default.visualize.synthetic_runtime(
    options_directory / "visualize.json"
)
visualize_options.to_json(options_directory / "visualize.json")

2026-06-04 23:34:26,453 - thuner.option.grid - WARNING - altitude not specified. Using default altitudes.
2026-06-04 23:34:26,454 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.


In [38]:
visualize_options.model_dump()

{'type': 'RuntimeOptions',
 'objects': {'convective': {'type': 'ObjectRuntimeOptions',
   'parent_local': PosixPath('/home/ewan/THUNER_output/runs/synthetic/geographic/options/visualize.json'),
   'style': 'presentation',
   'weights_filepath': None,
   'name': 'convective',
   'figures': [{'type': 'FigureOptions',
     'name': 'match',
     'function': 'thuner.visualize.runtime.visualize_tint_match',
     'style': 'presentation',
     'animate': True,
     'single_color': False,
     'template': None}],
   'animate': True,
   'single_color': False}}}

In [39]:
times = np.arange(
    np.datetime64(start),
    np.datetime64(end) + np.timedelta64(10, "m"),
    np.timedelta64(10, "m"),
)
track.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    visualize_options=visualize_options,
    output_directory=output_parent,
)

2026-06-04 23:34:27,846 - thuner.track.track - INFO - Beginning thuner tracking. Saving output to /home/ewan/THUNER_output/runs/synthetic/geographic.


2026-06-04 23:34:27,851 - thuner.track.track - INFO - Processing 2005-11-13T00:00:00.
2026-06-04 23:34:27,853 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:00:00.
2026-06-04 23:34:27,910 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-04 23:34:27,911 - thuner.track.track - INFO - Tracking convective.
2026-06-04 23:34:27,916 - thuner.match.match - INFO - Matching convective objects.
2026-06-04 23:34:27,917 - thuner.match.match - INFO - No current mask, or no objects in current mask.
2026-06-04 23:34:27,919 - thuner.visualize.runtime - INFO - Creating runtime visualization figures.
2026-06-04 23:34:29,560 - thuner.track.track - INFO - Processing 2005-11-13T00:10:00.
2026-06-04 23:34:29,561 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:10:00.
2026-06-04 23:34:30,888 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-04 23:34:30,889 - thuner.track.track - INFO - Tracking con

![THUNER applied to synthetic data.](https://raw.githubusercontent.com/THUNER-project/THUNER/refs/heads/main/gallery/synthetic_convective_20051113.gif)

'20051113'

In [42]:
gallery_directory = Path(track.__file__).parent.parent.parent / "gallery"
if gallery_directory.exists():
    gif_filename = f"convective_{format_time(start, day_only=True)}.gif"
    logger.info(f"Copying {gif_filename} to gallery.")
    gif_filepath = output_parent / f"visualize/match/{gif_filename}"
    shutil.copy(gif_filepath, gallery_directory / f"synthetic_{gif_filename}")
else:
    logger.warning("Gallery missing. Skipping GIF copy.")

2026-06-04 23:36:52,430 - __main__ - INFO - Copying convective_20051113.gif to gallery.


## Cartesian Coordinates

In [6]:
central_latitude = -10
central_longitude = 132

y = np.arange(-400e3, 400e3 + 2.5e3, 2.5e3).tolist()
x = np.arange(-400e3, 400e3 + 2.5e3, 2.5e3).tolist()

grid_options = option.grid.GridOptions(
    name="cartesian",
    x=x,
    y=y,
    central_latitude=central_latitude,
    central_longitude=central_longitude,
)
grid_options.to_json(options_directory / "grid.json")

2026-06-03 23:37:54,422 - thuner.option.grid - WARNING - altitude not specified. Using default altitudes.


In [ ]:
output_parent = base_local / "runs/synthetic/cartesian"
if output_parent.exists() & remove_existing_outputs:
    shutil.rmtree(output_parent)
    
times = np.arange(
    np.datetime64(start),
    np.datetime64(end) + np.timedelta64(10, "m"),
    +np.timedelta64(10, "m"),
)

track.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    visualize_options=None,
    output_directory=output_parent,
)

2026-06-03 23:37:55,143 - thuner.track.track - INFO - Beginning thuner tracking. Saving output to /home/ewan/THUNER_output/runs/synthetic/cartesian.
2026-06-03 23:37:55,145 - thuner.track.track - INFO - Processing 2005-11-13T00:00:00.
2026-06-03 23:37:55,146 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:00:00.


2026-06-03 23:37:56,331 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-03 23:37:56,331 - thuner.track.track - INFO - Tracking convective.
2026-06-03 23:37:56,360 - thuner.match.match - INFO - Matching convective objects.
2026-06-03 23:37:56,360 - thuner.match.match - INFO - No current mask, or no objects in current mask.
2026-06-03 23:37:56,363 - thuner.visualize.runtime - INFO - Creating runtime visualization figures.
2026-06-03 23:37:58,238 - thuner.track.track - INFO - Processing 2005-11-13T00:10:00.
2026-06-03 23:37:58,239 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:10:00.
2026-06-03 23:37:59,290 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-03 23:37:59,290 - thuner.track.track - INFO - Tracking convective.
2026-06-03 23:37:59,293 - thuner.write.mask - INFO - Writing convective masks to /home/ewan/THUNER_output/runs/synthetic/cartesian/output.zarr::masks/convective.
2026-06-03 23:37:59,327 - thuner

In [ ]:
ground_truth = analyze.synthetic.write_ground_truth(
    output_parent, data_options=data_options, times=times, grid_options=grid_options
)
print(ground_truth["synthetic"].head(10).to_string())